# Cache and offline use

`get_data` stores results on disk as per-column monthly Parquet fragments, so re-running a notebook does not download again. `MangoClient(offline=True)` serves only from that cache and needs no network. This notebook uses a private cache directory, so it does not touch your own cache (`mango.cache.clear()` empties it).

In [ ]:
import tempfile

import space_mango as mango

demo_dir = tempfile.mkdtemp()
c = mango.MangoClient(cache_dir=demo_dir)
c.count("magnetosheath", bz_imf_max=-2, columns=["Time", "Np"])

`est_mb` is what a request without cache would transfer; `download_mb_estimate` is what the cached `get_data` would download (whole months not yet in the cache). It is usually larger than `est_mb` on a cold cache.

In [ ]:
r = c.get_data("magnetosheath", bz_imf_max=-2, columns=["Time", "Np"])
info = c.cache_info()
{"n_files": info["n_files"], "size_bytes": info["size_bytes"]}

Everything is now cached, so the same request would download nothing: `download_mb_estimate` is 0.

In [ ]:
c.count("magnetosheath", bz_imf_max=-2, columns=["Time", "Np"])["download_mb_estimate"]

In [ ]:
offline = mango.MangoClient(cache_dir=demo_dir, offline=True)
offline.get_data("magnetosheath", bz_imf_max=-2, columns=["Time", "Np"]).to_polars().height

Errors explain themselves: they name the problem and, where possible, the valid choices.

In [ ]:
try:
    mango.magnetosheath.get_data(spacecraft="MMS1")
except mango.UnknownSpacecraftError as e:
    print(e)